# LaGoVAD-STP — first-run diagnostics D-1 / D-2 (**CPU only, no training**)

Why did R4 / R6 read "no difference shown" (`outputs/v_final/stp/REPORTS/stp_readout.md`, 2026-11-02)?
Reading rules written **before** any diagnostic number: `core/docs/final/STP_BUILD.md` §5. Printed only —
neither step changes §3's verdict; they decide what the next arm may be.

| step | question | input | output |
|---|---|---|---|
| **D-1** `stp_diag prompt` | Did the prompt train? R6's prefix `prompt_norm` moved 1.2612 → 1.2616 in 1,749 steps. Rebuild θ₀ (seed → `KATVAD.from_config`, fingerprint vs the step-0 `prompt_norm`), read AdamW's `exp_avg` / `exp_avg_sq` → **dead / noise / LR-bound / moved** | R4, R6 `s2024` run dirs | `REPORTS/stp_diag_prompt.md` |
| **D-2** `stp_diag pointing` | Do the top-12 Mo patches point at the anomaly, or at the frame border (ego-motion)? R4 − R1 was +0.012 on ego-involved clips, −0.013 on the others | DoTA-CAP-dev patch cache (`idx/`, `mo/`), window cache (ego-compensated Mo = R7), DoTA boxes | `REPORTS/stp_diag_pointing.md` |

**Runtime:** CPU runtime is enough (≈ 10–20 min, mostly Drive reads + one CLIP text-tower download).
**Upload first:** `core/tools/stp_diag.py`, `core/constants.py`, `core/docs/final/STP_BUILD.md` into `Thesis-Final/kat-vad/`.
**Then paste both `.md` files to Claude.**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip (DoTA_s1_ncc)
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + STP caches + outputs
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert (REPO / 'core/tools/stp_diag.py').is_file(), 'this upload has no core/tools/stp_diag.py -- re-upload'
assert hasattr(constants, 'STP_DIAG_INIT_NORM_RTOL'), 'constants.py predates the diagnostics -- re-upload'
assert '## 5.' in (REPO / 'core/docs/final/STP_BUILD.md').read_text(), 'STP_BUILD.md has no §5 (the reading rules)'

from core.data import stp_patches as stp_data      # noqa: E402
from core.tools import clip_windows               # noqa: E402

SEED     = constants.STP_SEEDS[0]                 # s2024, the only seed trained
ARMS     = ['R4', 'R6']
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
STRIDE_D, K, MOTION = constants.STP_DOTA_STRIDE, constants.STP_TOP_K, constants.STP_MOTION_RAW
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
WIN_D    = clip_windows.cache_dir(constants.DOTA_CAP_DATASET, STRIDE_D, constants.STP_WINDOW_GRIDS,
                                  constants.STP_CELLS_RESIZED)
PATCH_D  = stp_data.cache_dir(constants.DOTA_CAP_DATASET, STRIDE_D, MOTION, K)
ROOT     = constants.OUTPUT_ROOT / constants.STP_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/stp_diag')
LOCAL.mkdir(parents=True, exist_ok=True)

RUNS = {a: ROOT / a / f's{SEED}' for a in ARMS}
for a, d in RUNS.items():
    rec = json.loads((d / 'verified.json').read_text())
    assert rec['ckpt_bytes'] == (d / 'checkpoint_last.pt').stat().st_size, f'{a}: ckpt != verified.json'
    for f in ('config.yaml', 'metrics.jsonl'):
        assert (d / f).is_file(), f'{a}: no {f} in {d}'
    print(f'  {a:3s} step {rec["step"]}  {d}')
for name, path in (('DoTA s1', DOTA_S1), ('DoTA metadata', DOTA / 'metadata_val.json'),
                   ('DoTA split', DOTA / 'val_split.txt'), ('patch cache', PATCH_D / 'idx'),
                   ('patch cache', PATCH_D / 'mo'), ('window cache', WIN_D)):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:14s} {path}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn xlrd
python -c "import transformers; print('transformers', transformers.__version__)"

In [ ]:
ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd):
    # Stream a child's output here; raise with its tail on failure.
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}')


def stage_dir(src, dst):
    # Drive dir -> VM disk once (bit-identical copy, atomic rename).
    if dst.exists():
        return dst
    t0, tmp = time.time(), dst.with_name(dst.name + '.partial')
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp)
    os.replace(tmp, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    # VM-local file -> Drive atomically (.partial, fsync, rename; C10).
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)

## D-1 — did the prompt train? (`stp_diag prompt`)

Each run dir is copied to VM disk (ckpt ≈ 0.48 GB, `config.yaml`, `metrics.jsonl`). The tool rebuilds the
model with the run's own `config.yaml` and seed exactly as `core.train.main` does, so θ₀ is the real init
— checked by its norm against the `prompt_norm` logged on batch 0 (column "rebuilt / logged"). If that
check says **NOT reproduced**, read only the Adam columns.

In [ ]:
args = []
for a, d in RUNS.items():
    local = LOCAL / 'runs' / a
    local.mkdir(parents=True, exist_ok=True)
    for f in ('config.yaml', 'metrics.jsonl', 'checkpoint_last.pt'):
        if not (local / f).exists() or (local / f).stat().st_size != (d / f).stat().st_size:
            shutil.copy2(d / f, local / f)
    args += ['--run', a, local]
run([sys.executable, '-m', 'core.tools.stp_diag', 'prompt', *args, '--out-dir', LOCAL / 'out'])
for f in ('stp_diag_prompt.json', 'stp_diag_prompt.md'):
    publish(LOCAL / 'out' / f, REPORTS / f)
print((LOCAL / 'out' / 'stp_diag_prompt.md').read_text())

## D-2 — do the top-12 motion patches point at the anomaly? (`stp_diag pointing`)

Boxes: the DoTA GitHub annotation zip (`ego_involve` + per-frame boxes), taken from Drive if present,
else downloaded (as v5 Stage 0b did). The window cache (≈ 1 GB) is staged to VM disk once and gives the
**ego-compensated** Mo (frame-mean change removed first = arm R7's selection) beside the raw one.

In [ ]:
ANN = LOCAL / 'annotations'
drive_ann = DOTA / 'raw_annotations' / 'annotations'
if not ANN.exists():
    if drive_ann.is_dir():
        stage_dir(drive_ann, ANN)
    else:
        subprocess.run(['bash', '-c', 'cd /content/stp_diag && curl -sSL -o ann.zip '
                        'https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip '
                        '&& unzip -q ann.zip && rm ann.zip'], check=True)
assert ANN.is_dir(), f'no annotations at {ANN}'
print(len(list(ANN.glob('*.json'))), 'annotation files')

win = stage_dir(WIN_D, LOCAL / 'windows')
patch = LOCAL / 'patch'
for sub in ('idx', 'mo'):
    stage_dir(PATCH_D / sub, patch / sub)
run([sys.executable, '-m', 'core.tools.stp_diag', 'pointing', '--patch-dir', patch, '--windows-dir', win,
     '--s1-dir', DOTA_S1, '--annotations-dir', ANN, '--metadata', DOTA / 'metadata_val.json',
     '--split-file', DOTA / 'val_split.txt', '--out-dir', LOCAL / 'out'])
for f in ('stp_diag_pointing.json', 'stp_diag_pointing.md'):
    publish(LOCAL / 'out' / f, REPORTS / f)
print((LOCAL / 'out' / 'stp_diag_pointing.md').read_text())

## Done → **paste `REPORTS/stp_diag_prompt.md` + `REPORTS/stp_diag_pointing.md` to Claude**

In [ ]:
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/stp_diag_prompt.md and REPORTS/stp_diag_pointing.md to Claude')